# Comparison: Elliptical Grip vs. Learnable Complex Polynomial Mapping

This notebook compares two initial mapping strategies for the Neural Form-Finding pipeline:

1. **Elliptical Grip** — a fixed analytic mapping (Shirley-Chiu disk projection). Only a global scale and translation are learnable. Serves as a baseline.
2. **Learnable Asymmetric-Roots Polynomial** — a differentiable complex polynomial mapping parameterised by roots and weights. Rich expressivity, fully optimised end-to-end.

Six physical protocols from `data/configs/poster/` are used. Each protocol defines identical boundary conditions and loads; only the mapping strategy differs.

## 0. Environment Setup

In [23]:
import os
import sys
import pickle
import numpy as np

os.environ["JAX_PLATFORM_NAME"] = "cpu"

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
SRC_PATH  = os.path.join(REPO_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp

print(f'JAX version: {jax.__version__}  |  backend: {jax.default_backend()}')

JAX version: 0.6.2  |  backend: cpu


In [24]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Publication-quality style
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        150,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'lines.linewidth':   1.5,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

COLORS = {
    'elliptical': '#4878CF',   # blue
    'complex':    '#D65F5F',   # red
}
LABELS = {
    'elliptical': 'Elliptical Grip (baseline)',
    'complex':    'Complex Polynomial (learned)',
}

## 1. Pipeline Imports

In [25]:
from types import SimpleNamespace

from topology.builder import build_tessellation
from problem.conditions import configure_tessellation
from problem.config import load_and_parse_config
from jax_backend.state import CentroidalState
from jax_backend.pipeline import forward_pipeline
from jax_backend.training.trainer import train_pipeline
from jax_backend.geometry import reconstruct_vertices, compute_face_areas
from utils.visualization import plot_tessellation

ModuleNotFoundError: No module named 'topology'

## 2. Experiment Configuration

In [ ]:
N_EXPERIMENTS = 6
EXPERIMENT_IDS = list(range(1, N_EXPERIMENTS + 1))

CONFIG_BASE = os.path.join(REPO_ROOT, 'data', 'configs', 'poster')
CACHE_DIR   = os.path.join(REPO_ROOT, 'data', 'outputs', 'notebook_cache')
FIG_DIR     = os.path.join(REPO_ROOT, 'data', 'outputs', 'notebook_figures')

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FIG_DIR,   exist_ok=True)

def config_path(method: str, exp_id: int) -> str:
    return os.path.join(CONFIG_BASE, method, f'experience{exp_id}.yaml')

def cache_path(method: str, exp_id: int) -> str:
    return os.path.join(CACHE_DIR, f'{method}_exp{exp_id}.pkl')

# Elliptical-grip map params: only tx, ty, log_scale are learnable
ELLIPTICAL_MAP_PARAMS = {
    'tx':        jnp.array(0.0),
    'ty':        jnp.array(0.0),
    'log_scale': jnp.array(0.0),
}

print(f'Config root : {CONFIG_BASE}')
print(f'Cache dir   : {CACHE_DIR}')
print(f'Figures dir : {FIG_DIR}')

## 3. Training Helper

In [ ]:
def run_experiment(method: str, exp_id: int, force_rerun: bool = False) -> dict:
    """Run one (method, experiment) pair and return results dict.
    
    Results are cached to disk; set force_rerun=True to ignore the cache.
    """
    pkl = cache_path(method, exp_id)
    if not force_rerun and os.path.exists(pkl):
        print(f'  [{method}] exp{exp_id}: loading from cache')
        with open(pkl, 'rb') as f:
            return pickle.load(f)

    print(f'\n  [{method}] exp{exp_id}: training ...')

    # Always load from the 'complex' folder (identical physics); override map if needed.
    cfg_file = config_path('complex', exp_id)
    config   = load_and_parse_config(cfg_file)

    # Build tessellation
    topo = config.topology
    tessellation = build_tessellation(
        topo.get('pattern'),
        topo.get('width',  5),
        topo.get('height', 5),
    )
    requested_area = topo.get('total_area')
    if requested_area:
        import numpy as np_
        scale = np_.sqrt(requested_area / tessellation.compute_total_area())
        tessellation.update_vertices(tessellation.vertices * scale)

    configure_tessellation(tessellation, SimpleNamespace(**topo))
    initial_state = CentroidalState.from_tessellation(tessellation, target_cfg=config.target)

    # Select method-specific settings
    if method == 'elliptical':
        map_type         = 'elliptical_grip'
        initial_params   = ELLIPTICAL_MAP_PARAMS
        learn_global_scale = True   # log_scale is present in params
        use_shirley_chiu   = True
    else:  # complex
        map_type           = config.mapping.type
        initial_params     = config.mapping.params
        learn_global_scale = config.mapping.learn_global_scale
        use_shirley_chiu   = config.mapping.use_shirley_chiu
        # Inject log_scale if needed
        if learn_global_scale and 'log_scale' not in initial_params:
            initial_params = {**initial_params, 'log_scale': jnp.array(0.0)}

    optimized_params, history = train_pipeline(
        initial_params,
        initial_state,
        config.target,
        config.validity,
        config.physics,
        config.training,
        map_type           = map_type,
        use_shirley_chiu   = use_shirley_chiu,
        strict_boundary_fit= config.mapping.strict_boundary_fit,
        learn_global_scale = learn_global_scale,
    )

    # Final forward pass
    result = forward_pipeline(
        initial_state,
        config.target,
        config.validity,
        config.physics,
        map_type           = map_type,
        map_params         = optimized_params,
        use_shirley_chiu   = use_shirley_chiu,
        strict_boundary_fit= config.mapping.strict_boundary_fit,
    )

    # Collect scalar metrics from last epoch
    last = history[-1]
    final_metrics = {
        'chamfer':    float(last['chamfer_total']),
        'energy':     float(last['energy']),
        'stretch':    float(last['comp_phys_stretching']),
        'shear':      float(last['comp_phys_shearing']),
        'bending':    float(last['comp_phys_bending']),
        'contact':    float(last['comp_phys_contact']),
        'total_loss': float(last['total']),
        'area_dev':   float(last['global_material_area']),
    }

    # Serialize loss history as plain float lists (JAX arrays not pickle-safe)
    def _to_float(v):
        try:
            return float(v)
        except Exception:
            return None

    history_clean = [{k: _to_float(v) for k, v in ep.items() if not isinstance(v, dict)}
                     for ep in history]

    # Collect geometry arrays (convert to numpy for pickling)
    final_cnvs       = np.array(result['valid_state'].centroid_node_vectors)
    final_centroids  = np.array(result['valid_state'].face_centroids)
    displacements    = np.array(result['solution'].fields[-1])

    data = {
        'method':          method,
        'exp_id':          exp_id,
        'final_metrics':   final_metrics,
        'history':         history_clean,
        'final_cnvs':      final_cnvs,
        'final_centroids': final_centroids,
        'displacements':   displacements,
        'target':          {'type': config.target.type,
                            'center': list(config.target.center),
                            'radius': float(config.target.radius)},
    }

    with open(pkl, 'wb') as f:
        pickle.dump(data, f)
    print(f'  [{method}] exp{exp_id}: done. Total loss = {final_metrics["total_loss"]:.4e}')
    return data

## 4. Run All Experiments

Set `FORCE_RERUN = True` to ignore the cache and retrain from scratch.

In [ ]:
FORCE_RERUN = False  # Set True to re-train

results = {}  # (method, exp_id) -> data dict

for method in ['elliptical', 'complex']:
    for exp_id in EXPERIMENT_IDS:
        key = (method, exp_id)
        results[key] = run_experiment(method, exp_id, force_rerun=FORCE_RERUN)

print('\nAll experiments complete.')

## 5. Loss Convergence Curves

In [ ]:
def get_series(data: dict, key: str):
    return [ep.get(key) for ep in data['history'] if ep.get(key) is not None]

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
axes = axes.flatten()

for i, exp_id in enumerate(EXPERIMENT_IDS):
    ax = axes[i]
    for method in ['elliptical', 'complex']:
        data = results[(method, exp_id)]
        series = get_series(data, 'total')
        epochs = np.arange(len(series))
        ax.semilogy(epochs, series, color=COLORS[method], label=LABELS[method], alpha=0.9)
    ax.set_title(f'Experiment {exp_id}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Total loss (log scale)')
    if i == 0:
        ax.legend(frameon=False)

fig.suptitle('Training Convergence — Elliptical Grip vs. Complex Polynomial', fontsize=13, fontweight='bold')
fig.savefig(os.path.join(FIG_DIR, 'convergence_curves.pdf'))
fig.savefig(os.path.join(FIG_DIR, 'convergence_curves.png'))
plt.show()

## 6. Chamfer Distance Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
axes = axes.flatten()

for i, exp_id in enumerate(EXPERIMENT_IDS):
    ax = axes[i]
    for method in ['elliptical', 'complex']:
        data = results[(method, exp_id)]
        series = get_series(data, 'chamfer_total')
        epochs = np.arange(len(series))
        ax.semilogy(epochs, series, color=COLORS[method], label=LABELS[method], alpha=0.9)
    ax.set_title(f'Experiment {exp_id}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Chamfer distance (log)')
    if i == 0:
        ax.legend(frameon=False)

fig.suptitle('Chamfer Distance to Target Shape', fontsize=13, fontweight='bold')
fig.savefig(os.path.join(FIG_DIR, 'chamfer_curves.pdf'))
fig.savefig(os.path.join(FIG_DIR, 'chamfer_curves.png'))
plt.show()

## 7. Final Metric Bar Charts

In [ ]:
metric_keys  = ['chamfer', 'energy', 'stretch', 'shear', 'contact']
metric_labels = ['Chamfer distance', 'Total energy', 'Stretch energy', 'Shear energy', 'Contact energy']

x = np.arange(N_EXPERIMENTS)
width = 0.38

fig, axes = plt.subplots(1, len(metric_keys), figsize=(16, 4), constrained_layout=True)

for ax, key, label in zip(axes, metric_keys, metric_labels):
    ell_vals = [results[('elliptical', eid)]['final_metrics'][key] for eid in EXPERIMENT_IDS]
    cmp_vals = [results[('complex',    eid)]['final_metrics'][key] for eid in EXPERIMENT_IDS]

    bars_e = ax.bar(x - width / 2, ell_vals, width, color=COLORS['elliptical'], alpha=0.85, label=LABELS['elliptical'])
    bars_c = ax.bar(x + width / 2, cmp_vals, width, color=COLORS['complex'],    alpha=0.85, label=LABELS['complex'])

    ax.set_xticks(x)
    ax.set_xticklabels([f'Exp {i}' for i in EXPERIMENT_IDS], rotation=30, ha='right')
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Value')
    if ax is axes[0]:
        ax.legend(frameon=False, loc='upper left', fontsize=8)

fig.suptitle('Final Metric Comparison Across Experiments', fontsize=13, fontweight='bold')
fig.savefig(os.path.join(FIG_DIR, 'bar_metrics.pdf'))
fig.savefig(os.path.join(FIG_DIR, 'bar_metrics.png'))
plt.show()

## 8. Relative Improvement: Complex vs. Elliptical

In [ ]:
def pct_improvement(ell, cmp):
    """Percentage by which complex is better (lower is better for all metrics here)."""
    if abs(ell) < 1e-12:
        return 0.0
    return 100.0 * (ell - cmp) / abs(ell)

compare_keys = ['chamfer', 'energy', 'stretch', 'shear', 'contact', 'total_loss']
compare_labels = ['Chamfer', 'Energy', 'Stretch', 'Shear', 'Contact', 'Total loss']

x = np.arange(N_EXPERIMENTS)
width_bar = 0.13

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)

offsets = np.linspace(-(len(compare_keys) - 1) / 2, (len(compare_keys) - 1) / 2, len(compare_keys)) * width_bar
palette = plt.cm.Set2(np.linspace(0, 1, len(compare_keys)))

for j, (key, label, color, offset) in enumerate(zip(compare_keys, compare_labels, palette, offsets)):
    improvements = [
        pct_improvement(
            results[('elliptical', eid)]['final_metrics'][key],
            results[('complex',    eid)]['final_metrics'][key]
        )
        for eid in EXPERIMENT_IDS
    ]
    ax.bar(x + offset, improvements, width_bar, color=color, label=label, alpha=0.85)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels([f'Exp {i}' for i in EXPERIMENT_IDS])
ax.set_ylabel('Relative improvement (%) [positive = complex is better]')
ax.set_title('Relative Improvement of Complex Polynomial over Elliptical Grip', fontweight='bold')
ax.legend(frameon=False, loc='upper right', ncol=3)

fig.savefig(os.path.join(FIG_DIR, 'relative_improvement.pdf'))
fig.savefig(os.path.join(FIG_DIR, 'relative_improvement.png'))
plt.show()

## 9. Side-by-Side Final Shape Visualisation

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

def draw_structure(ax, cnvs, centroids, displacements, target_params, color='steelblue', title=''):
    """Draw the final deployed structure from cached geometry."""
    n_faces, max_nodes, _ = cnvs.shape

    final_centroids = centroids + displacements[:, :2]
    final_thetas    = displacements[:, 2]

    patches = []
    for f in range(n_faces):
        c  = final_centroids[f]
        th = final_thetas[f]
        R  = np.array([[np.cos(th), -np.sin(th)],
                        [np.sin(th),  np.cos(th)]])
        nodes = cnvs[f]               # (max_nodes, 2)
        verts = c + (R @ nodes.T).T   # rotated + translated
        # Remove zero-padded nodes
        norms = np.linalg.norm(nodes, axis=1)
        verts = verts[norms > 1e-10]
        if len(verts) >= 3:
            patches.append(MplPolygon(verts, closed=True))

    pc = PatchCollection(patches, facecolor=color, edgecolor='white', linewidth=0.5, alpha=0.75)
    ax.add_collection(pc)

    # Target circle / shape
    if target_params.get('type', 'circle') == 'circle':
        cx, cy = target_params['center']
        r      = target_params['radius']
        theta  = np.linspace(0, 2 * np.pi, 300)
        ax.plot(cx + r * np.cos(theta), cy + r * np.sin(theta),
                'k--', linewidth=0.8, alpha=0.6, label='Target')

    ax.set_aspect('equal')
    ax.autoscale_view()
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_axis_off()


fig, axes = plt.subplots(N_EXPERIMENTS, 2, figsize=(7, 3 * N_EXPERIMENTS), constrained_layout=True)

for i, exp_id in enumerate(EXPERIMENT_IDS):
    for j, method in enumerate(['elliptical', 'complex']):
        ax   = axes[i, j]
        data = results[(method, exp_id)]
        ch   = data['final_metrics']['chamfer']
        title = (f"{LABELS[method]}\n"
                 f"Exp {exp_id} | Chamfer = {ch:.4f}")
        draw_structure(
            ax,
            data['final_cnvs'],
            data['final_centroids'],
            data['displacements'],
            data['target'],
            color=COLORS[method],
            title=title,
        )

# Column headers
axes[0, 0].set_title(f'Elliptical Grip (baseline)\nExp 1 | Chamfer = {results[("elliptical", 1)]["final_metrics"]["chamfer"]:.4f}',
                     fontsize=9, fontweight='bold')

fig.suptitle('Final Deployed Configurations\n(left: elliptical grip  |  right: complex polynomial)',
             fontsize=12, fontweight='bold')

fig.savefig(os.path.join(FIG_DIR, 'final_shapes.pdf'))
fig.savefig(os.path.join(FIG_DIR, 'final_shapes.png'))
plt.show()

## 10. Quantitative Summary Table

In [ ]:
import pandas as pd

rows = []
for exp_id in EXPERIMENT_IDS:
    ell = results[('elliptical', exp_id)]['final_metrics']
    cmp = results[('complex',    exp_id)]['final_metrics']
    rows.append({
        'Experiment':    exp_id,
        'Method':        'Elliptical',
        'Chamfer':       ell['chamfer'],
        'Total energy':  ell['energy'],
        'Stretch':       ell['stretch'],
        'Shear':         ell['shear'],
        'Contact':       ell['contact'],
        'Total loss':    ell['total_loss'],
    })
    rows.append({
        'Experiment':    exp_id,
        'Method':        'Complex Poly.',
        'Chamfer':       cmp['chamfer'],
        'Total energy':  cmp['energy'],
        'Stretch':       cmp['stretch'],
        'Shear':         cmp['shear'],
        'Contact':       cmp['contact'],
        'Total loss':    cmp['total_loss'],
    })

df = pd.DataFrame(rows).set_index(['Experiment', 'Method'])

float_cols = ['Chamfer', 'Total energy', 'Stretch', 'Shear', 'Contact', 'Total loss']
styled = df.style.format({c: '{:.4e}' for c in float_cols}).highlight_min(
    subset=float_cols, axis=0, props='background-color: #d4edda; color: black;'
)
styled

In [ ]:
# Save as CSV and LaTeX
df.to_csv(os.path.join(FIG_DIR, 'summary_table.csv'))

latex = df.to_latex(
    float_format='%.4e',
    caption='Final metric comparison: elliptical grip vs.\ learnable complex polynomial mapping.',
    label='tab:comparison',
    column_format='llrrrrrr',
)
with open(os.path.join(FIG_DIR, 'summary_table.tex'), 'w') as f:
    f.write(latex)

print('LaTeX table saved to', os.path.join(FIG_DIR, 'summary_table.tex'))
print(latex)

## 11. Aggregate Summary (mean ± std over experiments)

In [ ]:
summary_rows = []
for method, label in [('elliptical', 'Elliptical Grip'), ('complex', 'Complex Polynomial')]:
    vals = {k: [] for k in ['chamfer', 'energy', 'stretch', 'shear', 'contact', 'total_loss']}
    for eid in EXPERIMENT_IDS:
        m = results[(method, eid)]['final_metrics']
        for k in vals:
            vals[k].append(m[k])
    row = {'Method': label}
    for k, v in vals.items():
        row[k + '_mean'] = np.mean(v)
        row[k + '_std']  = np.std(v)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Method')
df_summary

In [ ]:
# Radar / spider chart comparing the two methods
metric_radar = ['chamfer', 'stretch', 'shear', 'contact', 'energy']
labels_radar = ['Chamfer', 'Stretch', 'Shear', 'Contact', 'Total energy']
N_ax = len(metric_radar)

angles = np.linspace(0, 2 * np.pi, N_ax, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True), constrained_layout=True)

for method, label in [('elliptical', LABELS['elliptical']), ('complex', LABELS['complex'])]:
    values = [df_summary.loc[label if method == 'elliptical' else LABELS[method], f'{k}_mean']
              for k in metric_radar]
    # Normalise by elliptical mean to centre comparison on 1
    baseline = [df_summary.loc[LABELS['elliptical'], f'{k}_mean'] for k in metric_radar]
    normed = [v / b if b > 1e-12 else 0 for v, b in zip(values, baseline)]
    normed += normed[:1]
    ax.plot(angles, normed, color=COLORS[method], linewidth=2, label=label)
    ax.fill(angles, normed, color=COLORS[method], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_radar)
ax.set_yticks([0.25, 0.5, 0.75, 1.0, 1.25])
ax.set_yticklabels(['0.25x', '0.5x', '0.75x', '1x (baseline)', '1.25x'], fontsize=7)
ax.set_title('Normalised mean performance (smaller is better)', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), frameon=False)

fig.savefig(os.path.join(FIG_DIR, 'radar_chart.pdf'))
fig.savefig(os.path.join(FIG_DIR, 'radar_chart.png'))
plt.show()

---

**Figures saved to:** `data/outputs/notebook_figures/`
- `convergence_curves.{pdf,png}` — loss curves for all experiments
- `chamfer_curves.{pdf,png}` — chamfer distance convergence
- `bar_metrics.{pdf,png}` — final metric bar charts
- `relative_improvement.{pdf,png}` — % improvement of complex over elliptical
- `final_shapes.{pdf,png}` — side-by-side deployed configurations
- `radar_chart.{pdf,png}` — normalised aggregate performance
- `summary_table.{csv,tex}` — quantitative comparison table